In [1]:
!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta(cache_wheel_on_google_drive=False, serialization=False, mirror=0)
import pyrosetta; pyrosetta.init()
from pyrosetta import *
from pyrosetta.teaching import *


Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...

Resolving west.rosettacommons.org (west.rosettacommons.org)... 128.95.160.134, 128.95.160.135, 2607:4000:406::160:135, ...

HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
--2026-09-22 15:38:28--  https://west.rosettacommons.org/pyrosetta/quarterly/release/pyrosetta-0-cp313-cp313-linux_x86_64.whl
Reusing existing connection to west.rosettacommons.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1665598342 (1.6G) [application/octet-stream]
Saving to: ‘/PyRosetta/colab.bin/tmpbg75vcll/pyrosetta-0-cp313-cp313-linux_x86_64.whl’

pyrosetta-0-cp313-c 100%[===================>]   1.55G  4.47MB/s    in 2m 40s  

2026-09-22 15:41:09 (9.90 MB/s) - ‘/PyRos

In [3]:
import pyrosetta

pyrosetta.init(
    "-mute all"
)

print("PyRosetta initialized")
print(pyrosetta.version())

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.ubuntu 2026.29+releasequarterly.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python313.ubuntu r2 2026.29+releasequarter

In [28]:
from google.colab import files
from pathlib import Path
import shutil
import re

# Create a clean directory for this run
pdb_dir = Path("pdb_inputs")

if pdb_dir.exists():
    shutil.rmtree(pdb_dir)

pdb_dir.mkdir()

# Upload PDB files
uploaded = files.upload()

# Move and rename uploaded PDB files
pdb_files = []

for old_name in uploaded.keys():

    if not old_name.lower().endswith(".pdb"):
        continue

    # Example:
    # threaded_model.relaxed (1).pdb
    # -> threaded_model.relaxed_1.pdb
    clean_name = re.sub(r"\s*\((\d+)\)", r"_\1", old_name)

    # Replace remaining spaces
    clean_name = clean_name.replace(" ", "_")

    new_path = pdb_dir / clean_name

    # Move the uploaded file into the clean directory
    shutil.move(old_name, new_path)

    pdb_files.append(new_path)

print(f"\nFound {len(pdb_files)} PDB files:")

for pdb in sorted(pdb_files):
    print(" ", pdb)

Saving threaded_model.relaxed.pdb to threaded_model.relaxed (2).pdb
Saving 1AB1.relaxed.best.pdb to 1AB1.relaxed.best (3).pdb

Found 2 PDB files:
  pdb_inputs/1AB1.relaxed.best_3.pdb
  pdb_inputs/threaded_model.relaxed_2.pdb


In [30]:
xml_script = """
<ROSETTASCRIPTS>

  <SCOREFXNS>

    <ScoreFunction name="beta" weights="ref2015">
    </ScoreFunction>

  </SCOREFXNS>

  <RESIDUE_SELECTORS>

    <Layer
        name="surface"
        select_core="false"
        select_boundary="false"
        select_surface="true"
        use_sidechain_neighbors="true"/>

  </RESIDUE_SELECTORS>

  <TASKOPERATIONS>
  </TASKOPERATIONS>

  <FILTERS>

    <ResidueCount name="nres" />

    <PackStat
        name="packstat"
        threshold="0.0"
        confidence="0.0"/>

    <OversaturatedHbondAcceptorFilter
        name="oversat"
        scorefxn="beta"
        max_allowed_oversaturated="10"
        consider_mainchain_only="false"
        confidence="0.0"/>

    <CavityVolume
        name="cav"
        confidence="0.0"/>

    <ExposedHydrophobics
        name="exphyd"
        confidence="0.0"/>

    <BuriedUnsatHbonds
        name="new_buns_bb_heavy"
        report_bb_heavy_atom_unsats="true"
        scorefxn="beta"
        cutoff="40"
        residue_surface_cutoff="20.0"
        ignore_surface_res="true"
        print_out_info_to_pdb="true"/>

    <NetCharge name="netcharge" />

  </FILTERS>

  <SIMPLE_METRICS>

    <PerResidueSapScoreMetric
        name="perres_surface_SAP"
        residue_selector="surface"
        score_selector="surface" />

  </SIMPLE_METRICS>

  <MOVERS>

    <RunSimpleMetrics
        name="run_metrics3"
        metrics="perres_surface_SAP"
        prefix="surfaceSAP_"/>

  </MOVERS>

  <PROTOCOLS>

    <Add mover_name="run_metrics3"/>
    <Add filter_name="nres"/>
    <Add filter_name="netcharge"/>
    <Add filter_name="cav"/>
    <Add filter_name="new_buns_bb_heavy"/>
    <Add filter_name="exphyd"/>
    <Add filter_name="packstat"/>
    <Add filter_name="oversat"/>

  </PROTOCOLS>

  <OUTPUT/>

</ROSETTASCRIPTS>
"""

In [31]:
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

xml_objects = XmlObjects.create_from_string(xml_script)

print("Filters found:")
for name in xml_objects.list_filters():
    print(" -", name)

Filters found:
 - cav
 - exphyd
 - false_filter
 - netcharge
 - new_buns_bb_heavy
 - nres
 - oversat
 - packstat
 - true_filter


In [18]:
expected_filters = [
    "nres",
    "netcharge",
    "cav",
    "new_buns_bb_heavy",
    "exphyd",
    "packstat",
    "oversat",
]

for name in expected_filters:
    xml_objects.get_filter(name)

print("All requested filters were found.")

All requested filters were found.


In [34]:
def calculate_metrics(pdb_file, xml_objects, metric_names):

    # PyRosetta requires a string, not a pathlib Path
    pdb_filename = str(pdb_file)

    print(f"Loading: {pdb_filename}")

    # Load structure
    pose = pyrosetta.pose_from_pdb(pdb_filename)

    # Start result row
    results = {
        "pdb": Path(pdb_filename).name
    }

    # Calculate metrics
    for name in metric_names:
        filt = xml_objects.get_filter(name)
        results[name] = filt.report_sm(pose)

    return results

In [38]:
all_results = []

for pdb_file in sorted(pdb_files):

    print(f"Processing: {pdb_file.name}")

    result = calculate_metrics(
        pdb_file,
        xml_objects,
        metric_filter_names
    )

    all_results.append(result)

print(f"\nNumber of result rows: {len(all_results)}")

Processing: 1AB1.relaxed.best_3.pdb
Loading: pdb_inputs/1AB1.relaxed.best_3.pdb
Processing: threaded_model.relaxed_2.pdb
Loading: pdb_inputs/threaded_model.relaxed_2.pdb

Number of result rows: 2


In [40]:
results_df = pd.DataFrame(all_results)

print(f"Number of rows in table: {len(results_df)}")

display(results_df)

Number of rows in table: 2


,pdb,nres,netcharge,cav,new_buns_bb_heavy,exphyd,packstat,oversat
0,1AB1.relaxed.best_3.pdb,46.0,0.0,0.00000,2.0,745.397110,0.680549,0.0
1,threaded_model.relaxed_2.pdb,182.0,-2.0,264.30846,10.0,2594.195689,0.476021,1.0


In [42]:
output_csv = "rosetta_metrics.csv"

results_df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")

Saved: rosetta_metrics.csv


In [43]:
from google.colab import files

files.download(output_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>